<a href="https://colab.research.google.com/github/Glaze0/Assignment/blob/main/Copy_of_advanced_rag_chunking_retrieval_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced RAG: Chunking & Retrieval HandsOn

### Overview
This notebook covers the two preparation steps that run **before any question is ever asked**: getting clean text out of documents and splitting it effectively.


---
## 0. Setup

In [ ]:
%pip install -q sentence-transformers langchain-text-splitters tiktoken beautifulsoup4 pypdf reportlab scikit-learn matplotlib

In [4]:
import numpy as np
import re   #regular expression
import time
import sqlite3
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# Configure output display
np.set_printoptions(precision=3, suppress=True)   #Controls how many digits are shown after the decimal point when  .



# Initialize model
EMB = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def encode(texts, **kwargs):
    """Encodes text using the MiniLM model."""
    return EMB.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False,
        **kwargs
        )


print("ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ready


## 1. Extraction vs. Parsing

The difference between extraction and parsing is significant for RAG quality:

*   **Extraction:** Pulls raw characters off the page. You get text, but lose context of what the text *is*.
*   **Parsing:** Labels each block (heading, table row, caption) and preserves their relationships.

> **Key Concept:** Structure is as valuable as the text. Knowing a number belongs to a specific table row rather than loose prose allows the model to interpret it correctly.

In [ ]:
# The same information, extracted flat vs parsed with structure.
flat_text = """Q3 Financial Summary
Region Revenue Growth
North 4200000 12
South 3100000 -3
East 5600000 21
Notes: growth is year over year."""

parsed = {
    "title": "Q3 Financial Summary",
    "table": {
        "columns": ["Region", "Revenue", "Growth %"],
        "rows": [
            ["North", 4_200_000, 12],
            ["South", 3_100_000, -3],
            ["East", 5_600_000, 21]
        ],
    },
    "note": "growth is year over year",
}

print("FLAT — what does 21 mean? Which column is 3100000 in?")
print(flat_text)
print("\nPARSED — every number has a row, a column, and a unit:")
print(parsed["table"]["columns"], parsed["table"]["rows"][2])

FLAT — what does 21 mean? Which column is 3100000 in?
Q3 Financial Summary
Region Revenue Growth
North 4200000 12
South 3100000 -3
East 5600000 21
Notes: growth is year over year.

PARSED — every number has a row, a column, and a unit:
['Region', 'Revenue', 'Growth %'] ['East', 5600000, 21]


In [ ]:
# Now ask a question of each, via retrieval.
query = encode("Which region grew fastest?")

flat_chunks = [line.strip() for line in flat_text.split("\n") if line.strip()]

structured_chunks = [
    f"{r[0]} region: revenue {r[1]: ,} USD, year-over-year growth {r[2]} %"
    for r in parsed["table"]["rows"]
]

for label, chunks in [("FLAT", flat_chunks), ("STRUCTURED", structured_chunks)]:
    embeddings = encode(chunks)
    scores = embeddings @ query
    best_index = int(np.argmax(scores))
    print(f"{label:<12} top-chunk -> {chunks[best_index]!r}")

FLAT         top-chunk -> 'Region Revenue Growth'
STRUCTURED   top-chunk -> 'North region: revenue  4,200,000 USD, year-over-year growth 12 %'


### The Result
The **FLAT** version retrieves a naked row of numbers. Even if retrieval works, the LLM receives `South 3100000 -3` and must guess meanings. The **STRUCTURED** version retrieves a self-describing sentence.


## 2. Know your format, route your tool

Effective ingestion requires routing rules based on file types. Let's look at the most common web format: HTML.

In [ ]:
from bs4 import BeautifulSoup

html = """
<html><head><title>Leave Policy</title><script>var x=1;</script><style>.nav{color:red}</style></head><body>
  <nav><a href=\"/\">Home</a><a href=\"/hr\">HR</a><a href=\"/it\">IT</a></nav>
  <header><h1>Acme Corp Intranet</h1></header>
  <main>
    <h2>Annual Leave</h2>
    <p>Employees accrue 18 days of paid leave annually, pro-rated for part-time staff.</p>
    <h2>Carry Over</h2>
    <p>Up to 5 unused days may be carried into the following year.</p>
    <table><tr><th>Grade</th><th>Days</th></tr><tr><td>Junior</td><td>18</td></tr>
    <tr><td>Senior</td><td>24</td></tr></table>
  </main>
  <footer>&copy; 2026 Acme Corp. Privacy. Terms. Cookies.</footer></body></html>"""

soup = BeautifulSoup(html, "html.parser")
naive = soup.get_text(separator=" ", strip=True)

print("NAIVE get_text():\n", naive[:220], "\n")
print(f"chars: {len(naive)}")

NAIVE get_text():
 Leave Policy Home HR IT Acme Corp Intranet Annual Leave Employees accrue 18 days of paid leave annually, pro-rated for part-time staff. Carry Over Up to 5 unused days may be carried into the following year. Grade Days Ju 

chars: 280


In [ ]:
from bs4 import BeautifulSoup

# Targeted extraction: drop the boilerplate, keep the structure as you go.
soup2 = BeautifulSoup(html, "html.parser")

# Remove non-content elements
for tag in soup2(["script", "style", "nav", "footer", "header"]):
    tag.decompose()

blocks = []
main_content = soup2.find("main") or soup2
current_heading = None

for el in main_content.find_all(["h1", "h2", "h3", "p", "table"]):
    if el.name.startswith("h"):
        current_heading = el.get_text(strip=True)
    elif el.name == "p":
        blocks.append({"type": "paragraph", "text": el.text.strip(), "heading": current_heading})
    elif el.name == "table":
        rows = [[c.get_text(strip=True) for c in tr.find_all(["td", "th"])]
                for tr in el.find_all("tr")]
        header = rows[0]
        body = rows[1:]
        for r in body:
            sentence = ", ".join(f"{h} : {v}" for h, v in zip(header, r))
            blocks.append({"type": "table_row", "text": sentence, "heading": current_heading})

for b in blocks:
    print(f"[{b["type"]:<10}] ({b["heading"]}) {b["text"]}")

[paragraph ] (Annual Leave) Employees accrue 18 days of paid leave annually, pro-rated for part-time staff.
[paragraph ] (Carry Over) Up to 5 unused days may be carried into the following year.
[table_row ] (Carry Over) Grade : Junior, Days : 18
[table_row ] (Carry Over) Grade : Senior, Days : 24


## 3. Probe before you parse

This is the **#1 silent trap** in RAG pipelines: an image-only PDF (scanned) often returns an empty string without raising an error. You index nothing, and the system fails silently.

Let's demonstrate how standard parsers fail on scanned documents compared to text-layer PDFs.

In [ ]:
import os
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from PIL import Image, ImageDraw

os.makedirs("/tmp/pdfs", exist_ok=True)

# (a) A normal, text-layer PDF
c = canvas.Canvas("/tmp/pdfs/text_layer.pdf", pagesize=letter)
c.drawString(72, 720, "Employees accrue 18 days of paid leave annually.")
c.drawString(72, 700, "Up to 5 unused days may be carried into the following year.")
c.showPage()
c.save()

# (b) A "scanned" PDF: rendered as an IMAGE
img = Image.new("RGB", (1200, 300), "white")
draw = ImageDraw.Draw(img)
draw.text((40, 120), "Employees accrue 18 days of paid leave annually.", fill="black")
img.save("/tmp/pdfs/page.png")    #created an image

c = canvas.Canvas("/tmp/pdfs/scanned.pdf", pagesize=letter)
c.drawImage("/tmp/pdfs/page.png", 50, 500, width=500, height=125)   # Assing previously created image to pdf
c.showPage()
c.save()

print(f"created: {os.listdir('/tmp/pdfs')}")

created: ['text_layer.pdf', 'page.png', 'scanned.pdf']


In [ ]:
import pypdf
for name in ["text_layer.pdf", "scanned.pdf"]:
    r = pypdf.PdfReader(f"/tmp/pdfs/{name}")
    text = r.pages[0].extract_text()
    print(f"{name:<16} -> {len(text):>3} chars extracted   {text[:52]!r}")

text_layer.pdf   -> 109 chars extracted   'Employees accrue 18 days of paid leave annually.\nUp '
scanned.pdf      ->   0 chars extracted   ''


### The Silent Failure
Observe the output above. There is **no exception or warning**. A naive pipeline would embed an empty string and move on.


In [ ]:
def has_text_layer(path, sample_pages=3, min_chars=20):
    """Check the first few pages for any real text."""
    reader = pypdf.PdfReader(path)
    for page in reader.pages[:sample_pages]:
        text = (page.extract_text() or "").strip()
        if len(text) >= min_chars:
            return True
    return False


def parse_with_pypdf(path):
    """Extract text using pypdf."""
    reader = pypdf.PdfReader(path)
    return "\n".join((p.extract_text() or "") for p in reader.pages)


def run_ocr(path):
    """Placeholder for OCR engine."""
    return "[OCR would run here - tesseract, or a VLM for complex layouts]"


for name in ["text_layer.pdf", "scanned.pdf"]:
    file_path = f"/tmp/pdfs/{name}"
    route = "pypdf" if has_text_layer(file_path) else "OCR"

    if route == "pypdf":
        output = parse_with_pypdf(file_path)
    else:
        output = run_ocr(file_path)

    print(f"{name:<16} route={route:<6} -> {output.strip()[:56]!r}")

text_layer.pdf   route=pypdf  -> 'Employees accrue 18 days of paid leave annually.\nUp to 5'
scanned.pdf      route=OCR    -> '[OCR would run here - tesseract, or a VLM for complex la'


## Tables: Query them, don't embed them

Aggregations, rankings, and exact filters are **computations**. Similarity search cannot perform them accurately, regardless of the embedding quality.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute(
    "CREATE TABLE sales (region TEXT, product TEXT, revenue INTEGER, quarter TEXT)"
)

sales_data = [
    ("North", "Widget", 4_200_000, "Q3"),
    ("South", "Widget", 3_100_000, "Q3"),
    ("East", "Widget", 5_600_000, "Q3"),
    ("West", "Widget", 2_900_000, "Q3"),
    ("North", "Gadget", 1_800_000, "Q3"),
    ("South", "Gadget", 2_400_000, "Q3"),
    ("East", "Gadget", 900_000, "Q3"),
    ("West", "Gadget", 3_300_000, "Q3"),
    ("North", "Widget", 3_900_000, "Q2"),
    ("East", "Gadget", 1_100_000, "Q2"),
]

conn.executemany("INSERT INTO sales VALUES (?,?,?,?)", sales_data)
conn.commit()

row_count = conn.execute("SELECT COUNT(*) FROM sales").fetchone()[0]
print(f"{row_count} rows inserted.")

10 rows inserted.


In [ ]:
import numpy as np

# THE WRONG WAY: embed each row as a sentence, retrieve by similarity.
# Use sales_data instead of rows to get (region, product, revenue, quarter)
row_texts = [
    f"{region} sold {product} for {revenue:,} USD {quarter}"
    for region, product, revenue, quarter in sales_data
]
ROW_V = encode(row_texts)
question = "What was the total Q3 revenue across all regions?"
q_vec = encode(question)

# Calculate the similarity scores
scores = ROW_V @ q_vec
top_indices = np.argsort(-scores)[:3]   # -scores is used to sort element in descending order of scores

print("Vector search returned:")
for i in top_indices:
    print(f"   {row_texts[i]}")

print("\n-> The LLM now has 3 of 10 rows and is asked to sum. It cannot see the other 7.")

Vector search returned:
   East sold Gadget for 900,000 USD Q3
   West sold Gadget for 3,300,000 USD Q3
   North sold Gadget for 1,800,000 USD Q3

-> The LLM now has 3 of 10 rows and is asked to sum. It cannot see the other 7.


In [ ]:
import sqlite3

# THE RIGHT WAY: text-to-SQL. Retrieve the SCHEMA, not the rows.

schema = """
TABLE sales
    region TEXT         -- one of : North, South, East and West
    product TEXT        -- one of :Widget, Gadget
    revenue INTEGER     -- USD
    quarter TEXT        -- one of : Q1, Q2, Q3, Q4
"""
# In production an LLM writes the following sql from the above schema + the question (text-to-sql)
generated_sql = "SELECT SUM(revenue) FROM sales WHERE quarter = 'Q3'"
result = conn.execute(generated_sql).fetchall()
print(result)


[(24200000,)]


In [ ]:
# The three question types slide 8 lists, each impossible for similarity search.
queries = [
    ("aggregation  ", "SELECT region, SUM(revenue) FROM sales WHERE quarter='Q3' GROUP BY region"),
    ("ranking      ", "SELECT region, product, revenue FROM sales ORDER BY revenue DESC LIMIT 3"),
    ("exact filter ", "SELECT * FROM sales WHERE quarter='Q2' AND revenue > 2000000"),
]

for label, sql in queries:
    res = conn.execute(sql).fetchall()
    print(f"{label} {res}")

aggregation   [('East', 6500000), ('North', 6000000), ('South', 5500000), ('West', 6200000)]
ranking       [('East', 'Widget', 5600000), ('North', 'Widget', 4200000), ('North', 'Widget', 3900000)]
exact filter  [('North', 'Widget', 3900000, 'Q2')]


## The Chunk is the Unit of Retrieval

RAG retrieves **chunks**, not documents. Your splitting strategy determines the limits of what can be found.

First, let's establish a corpus and a small evaluation set to measure performance.

In [1]:
DOCUMENT = """# Acme Corp Employee Handbook

## Annual Leave
Employees accrue 18 days of paid leave annually, pro-rated for part-time staff. Leave accrues
monthly from the start date. Up to 5 unused days may be carried into the following year; anything
beyond 5 days is forfeited on 31 December.

## Remote Work
Remote work requires manager approval and a signed home-office checklist. Employees may work
remotely up to 3 days per week. Fully remote arrangements need director-level sign-off and a
review after six months.

## Expenses
Expense reports must be filed within 30 days of travel. Receipts are required for any item over
25 USD. Reimbursement is processed in the payroll run following approval, so allow up to 6 weeks.
Mileage is reimbursed at 0.45 USD per mile.

## Equipment
New hires receive a laptop and monitor within their first week. Replacement cycles are 3 years for
laptops and 5 years for monitors. Damaged equipment should be reported to IT within 48 hours;
accidental damage is covered once per year without charge.

## Performance Reviews
The annual review cycle begins in March and concludes in May. Ratings are calibrated across teams
before being communicated. Promotion decisions are made in a separate June cycle and require a
written case from the manager.
"""

print(f"{len(DOCUMENT)} chars, {len(DOCUMENT.split())} words")

1274 chars, 210 words


In [ ]:
# A small evaluation set. THIS is what makes every later comparison meaningful.
# Each question is paired with a phrase that MUST appear in a retrieved chunk for it to count.
EVAL = [
    ("How many leave days do I get?", "18 days of paid leave"),
    ("Can I carry unused holiday forward?", "carried into the following year"),
    ("How many days a week can I work remotely?", "3 days per week"),
    ("What do I need for fully remote work?", "director-level sign-off"),
    ("How long do I have to submit expenses?", "within 30 days of travel"),
    ("Do I need a receipt for a 30 dollar meal?", "over\n25 USD"),
    ("What is the mileage rate?", "0.45 USD per mile"),
    ("How often is my laptop replaced?", "3 years for"),
    ("What if I break my laptop?", "within 48 hours"),
    ("When do performance reviews happen?", "begins in March"),
    ("When are promotions decided?", "June cycle"),
]

print(f"{len(EVAL)} labelled questions")

11 labelled questions


> **Important:** Never skip the evaluation set. Eleven hand-written questions turn opinions into **measurements**. "Start simple, measure, iterate" is impossible without data.

In [ ]:
def evaluate_chunks(chunks, eval_set=EVAL, k=3):
    """Retrieval hit rate: is the required phrase inside any top-k chunks?"""
    if not chunks:
        return 0.0, 0.0

    chunk_embeddings = encode(chunks)
    hits = []
    reciprocal_ranks = []

    for question, needle in eval_set:
        q_embedding = encode(question)
        order = np.argsort(-(chunk_embeddings @ q_embedding))[:k]

        def normalize(s):
            return re.sub(r"\s+", " ", s)

        found = [j for j, idx in enumerate(order)
                 if normalize(needle) in normalize(chunks[idx])]

        hits.append(1.0 if found else 0.0)
        reciprocal_ranks.append(1 / (found[0] + 1) if found else 0.0)

    return float(np.mean(hits)), float(np.mean(reciprocal_ranks))

## Splitter Strategies

### Fixed-size
The blunt baseline. It often cuts mid-word or mid-sentence, wrecking the semantic meaning of the vector.

In [ ]:
def chunk_fixed(text, size=400):
    """Splits text into fixed-size character chunks."""
    return [text[i : i + size] for i in range (0, len(text), size)]


fixed_chunks = chunk_fixed(DOCUMENT, 400)

print(f"{len(fixed_chunks)} chunks\n")
print("--- boundary between chunk 0 and 1 ---")
print(f"... {repr(fixed_chunks[0][-70:])}")
print(f"    {repr(fixed_chunks[1][:70])} ...")

4 chunks

--- boundary between chunk 0 and 1 ---
... 'roval and a signed home-office checklist. Employees may work\nremotely '
    'up to 3 days per week. Fully remote arrangements need director-level s' ...


Look at where it cut. Mid-word, or mid-sentence, or straight through a fact. The deck's example isexactly this: a blind cut leaves `"...convolutional architectures are supe"` — and **embeddingbroken meaning wrecks retrieval**, because the vector now represents a fragment that means nothing.
### Sliding window — fixed-size plus overlap

In [ ]:
def chunk_sliding(text, size=400, overlap=80):
    """Splits text into fixed-size chunks with a sliding window overlap."""
    step = size - overlap
    chunks = [
        text[i : i + size]
        for i in range(0, len(text) - size + 1, step)
        if text[i : i + size].strip()
    ]
    return chunks


sliding_chunks = chunk_sliding(DOCUMENT, 400, 80)

print(f"{len(sliding_chunks)} chunks (vs {len(fixed_chunks)} without overlap)")
print(f"\ntail of chunk 0: {repr(sliding_chunks[0][-60:])}")
print(f"head of chunk 1: {repr(sliding_chunks[1][:60])}  <- the overlap")

3 chunks (vs 4 without overlap)

tail of chunk 0: 'a signed home-office checklist. Employees may work\nremotely '
head of chunk 1: 'anager approval and a signed home-office checklist. Employee'  <- the overlap


### Sliding Window
Fixed-size plus **overlap**. This ensures that facts sitting on a boundary appear complete in at least one chunk. This is the "sane default."

In [5]:
def chunk_sentences(text, max_chars=400):
    """Splits text into chunks while respecting sentence boundaries."""
    # take home assignment
    chunks = []
    for i in range(0,len(text),max_chars):
      chunks.append(text[i:i+max_chars])


    return chunks


sent_chunks = chunk_sentences(DOCUMENT)
lengths = [len(c) for c in sent_chunks]

print(f"{len(sent_chunks)} chunks")
print(f"sizes: min {min(lengths)}, max {max(lengths)}, std {np.std(lengths):.0f}")

4 chunks
sizes: min 74, max 400, std 141


### Sentence-based
Keeps grammar intact, but results in highly uneven chunk sizes. Small chunks lack context; huge ones dilute meaning.

In [6]:
def chunk_paragraphs(text, max_chars=600):
    # take home sentence
    out = []
    for i in text.split('\n\n'):
      if len(i) < max_chars:
        out.append(i)
      else:
        for j in i.split('.'):
          if len(j) < max_chars:
            out.append(j)

    return out


### Paragraph-based
Respects natural thought boundaries.

> **Warning:** Always cap chunk size. A paragraph splitter without a cap is a truncation bug waiting to happen.

In [ ]:
def chunk_markdown(text):
    """Split on headers and prepend the header to its section."""
    # Split on headers, and PREPEND the header to its section so each chunk is self-describing.
    parts = re.split(r"\n(?=#{1,6}\s)", text.strip())
    out = []
    for p in parts:
        lines = p.strip().split("\n")
        header = lines[0].lstrip("#").strip() if lines[0].startswith("#") else None
        body = "\n".join(lines[1:]).strip() if header else p.strip()
        if not body:
            continue
        out.append({
            "text": f"{header}\n{body}" if header else body,
            "metadata": {"section": header}
        })
    return out


md_chunks = chunk_markdown(DOCUMENT)
for c in md_chunks:
    section_name = str(c['metadata']['section'])
    print(f"[{section_name:<20}] {len(c['text']):>3} chars  {c['text'][:44]}...")

### Structure-aware
Splits on document seams (like Markdown headers). This allows chunks to carry **metadata** (like section names), which is crucial for citations.

In [ ]:
def chunk_semantic(text, threshold_percentile=25, max_chars=700):
    """Splits text where cosine similarity between sentences dips."""
    # 1. Split into sentences
    clean_text = text.replace("\n", " ")
    sentences = [
        s.strip() for s in re.split(r"(?<=[.!?])\s+", clean_text)
        if len(s.strip()) > 15
    ]

    # 2. Embed each sentence
    v_embeddings = encode(sentences)

    # 3. Calculate similarities between neighbors
    similarities = np.array([
        float(v_embeddings[i] @ v_embeddings[i + 1])
        for i in range(len(v_embeddings) - 1)
    ])

    # 4. Determine boundaries
    cut_threshold = np.percentile(similarities, threshold_percentile)

    chunks = []
    current_chunk = [sentences[0]]

    for i, sentence in enumerate(sentences[1:]):
        content = " ".join(current_chunk)
        too_big = len(content) + len(sentence) > max_chars

        if similarities[i] < cut_threshold or too_big:
            chunks.append(content)
            current_chunk = []
        current_chunk.append(sentence)

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks, similarities


sem_chunks, sims = chunk_semantic(DOCUMENT)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

# Visualize the sentence-to-sentence similarity dips
ax.plot(sims, "o-", color="#3b7dd8", linewidth=1.5, markersize=5)
threshold = np.percentile(sims, 25)
ax.axhline(threshold, color="crimson", linestyle="--", label="cut threshold (25th pct)")

for i, s in enumerate(sims):
    if s < threshold:
        ax.axvline(i, color="crimson", alpha=0.2, linewidth=6)

ax.set_xlabel("boundary between sentence i and i+1")
ax.set_ylabel("cosine similarity")
ax.set_title("Semantic chunking: the dips are where the topic changes")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 6f. Semantic Chunking
Cuts where meaning shifts by monitoring dips in cosine similarity between sentences.

> **Trade-off:** High quality, but requires an embedding call for every sentence during ingestion. This can be expensive at scale.

### Take Home Assignment
Write the Recursive Chunking - we have all the pieces involved